# Tune DIODE hyperparameters on example data

Here we used the bundled synthetic data in data/data_example/. It runs cross-validation over a hyperparameter grid, refits the selected model, and evaluates the recovered network against the synthetic ground truth.

## Instructions

### How the input files fit together

The expression matrix has one **row per gene** and one **column per observation**, such as a cell or sampled time point. The bundled example has 10 genes and 50 observations. Here, GeneN stands for the final gene name, and tN labels the final observation. Values below come from that dataset and are rounded for display.

| Gene | t1 | t2 | t3 | t4 | … | tN |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| g1 (`Gene1`) | 0.6002 | 0.5964 | 0.5954 | 0.5470 | … | 0.1516 |
| g2 (`Gene2`) | 1.1481 | 1.1472 | 1.1025 | 1.1423 | … | 0.6416 |
| g3 (`Gene3`) | 1.0686 | 1.0412 | 1.0346 | 0.9364 | … | 0.3737 |
| g4 (`Gene4`) | 1.4326 | 1.4004 | 1.4003 | 1.3538 | … | 0.0975 |
| … | … | … | … | … | … | … |
| gN (`GeneN`) | 1.1914 | 1.2010 | 1.2711 | 1.2775 | … | 2.1113 |

The gene file labels rows in order. The pseudotime file lists 0.000000, 0.020408, 0.040816, 0.061224, …, 1.000000 in the same order as the columns. For example, the first expression value, **0.6002**, is the measurement for g1 (Gene1) at t1(pseudotime 0.000000). The value **1.1481** is for g2 at that same time, and **0.5964** is for g1 at t2. Each gene has a value at every observation. DIODE sorts the columns by pseudotime before fitting.

### Format your files

The excerpts below show the same rows and columns as the table. Save the complete values in each file, omit the ellipses, and use the actual final gene name in place of GeneN.

- **Expression (datamatrix.txt)**. Save a whitespace-separated numeric matrix with genes as rows and observations as columns. Omit the display labels and header.

  ```text
  0.6002  0.5964  0.5954  0.5470  …  0.1516
  1.1481  1.1472  1.1025  1.1423  …  0.6416
  1.0686  1.0412  1.0346  0.9364  …  0.3737
  1.4326  1.4004  1.4003  1.3538  …  0.0975
  …
  1.1914  1.2010  1.2711  1.2775  …  2.1113
  ```

- **Pseudotime (pseudotime.txt)**. Save one numeric value per line and keep those lines in the same order as the expression columns. The number of lines must equal the number of columns.

  ```text
  0.000000
  0.020408
  0.040816
  0.061224
  …
  1.000000
  ```

- **Genes (genes.txt)**. Save one gene name per line in expression row order. The example has 10 genes and 50 pseudotime values.

  ```text
  Gene1
  Gene2
  Gene3
  Gene4
  …
  GeneN
  ```

- **Reference network (reference_network.csv)**. To evaluate network recovery, supply a square gene-by-gene matrix with target genes as rows and source regulators as columns, both in the gene-list order. The bundled example includes a binary reference and a signed reference in reference_network_signed.csv.

Run this notebook from the Example/ directory or the repository root. To use your own data, copy configs/data_example.yaml, set the expression, pseudotime, gene, and output paths for your dataset, and point config_path below at your copy.


In [10]:
from pathlib import Path

import yaml

from diode import DIODE

repo_root = Path.cwd()
if repo_root.name == "Example":
    repo_root = repo_root.parent

config_path = repo_root / "configs/data_example.yaml"

## Set the hyperparameter grid

Edit the candidate lists below to tune DIODE. The cell overwrites solvers.diode.hyperparameter_grid in configs/data_example.yaml. The training defaults live in the YAML file.

In [ ]:
# Define the hyperparameter grid for DIODE. This is a list of candidate values to try during hyperparameter tuning.
hyperparameter_grid = {
    "huber_delta": [0.01, 2.0],
    "l2_lambda": [1e-07, 1e-05],
}

# Write the hyperparameter grid to the config file.
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
config["solvers"]["diode"]["hyperparameter_grid"] = hyperparameter_grid
with config_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False, default_flow_style=None)

## Fit DIODE

The fit() call runs the same workflow used by the dataset runner: cross-validation over the hyperparameter grid, selection by validation error and rank stability, final refit, and perturbation scoring.

In [7]:
model = DIODE(config_path)
result = model.fit()

print("Output directory:", result["output_dir"])
print("Selected hyperparameters:")
print({
    key: result["metrics"][key]
    for key in hyperparameter_grid
})

[001/004] hd_0.01_l2_1e-07_tc_0.0001
initial train_total=3.48884251e-03 train_data=3.48883251e-03 val_data=3.46026372e-03
epoch 0000 train=3.48883251e-03 val=3.46026372e-03 grad_max=7.709e-04 step_max=nan loss_change=nan dA=0.000e+00 dX*=0.000e+00 best_val=3.46026372e-03@0 warning=internal LBFGS gradient tolerance was already met; this epoch may have had no effective parameter update
epoch 0004 train=3.48883251e-03 val=3.46026372e-03 grad_max=7.709e-04 step_max=nan loss_change=nan dA=0.000e+00 dX*=0.000e+00 best_val=3.46026372e-03@0 stop=parameter_change (delta_A_max_abs and delta_X_star_max_abs) <= stop_tolerance_change for 5 consecutive epochs warning=internal LBFGS gradient tolerance was already met; this epoch may have had no effective parameter update
initial train_total=3.66806902e-03 train_data=3.66805902e-03 val_data=3.46026372e-03
epoch 0000 train=3.66805902e-03 val=3.46026372e-03 grad_max=8.124e-04 step_max=nan loss_change=nan dA=0.000e+00 dX*=0.000e+00 best_val=3.46026372e-0

## Optional evaluation with ground truth

DIODE can fit using expression, pseudotime, and gene names. To evaluate against ground truth, set `reference_network_dir` in the YAML and put `reference_network.csv` in that directory. The `evaluate()` call uses that reference to compute perturbation AUROC.


In [ ]:
eval_metrics = model.evaluate()
print(f"AUROC: {eval_metrics['auroc_perturb']:.3f}")

AUROC: 0.588
